<a href="https://colab.research.google.com/github/alearecuest/anyoneai-exercises-sprint_3/blob/main/3_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangChain: Memory

### **Notebook Overview**

This notebook explores **conversational memory** in LangChain, enabling chatbots and agents to maintain context across multiple interactions. You will learn different memory strategies to handle conversation history efficiently while managing token limits.

### **What You Will Learn**

* Why LLMs are stateless and need external memory
* Implementing ConversationBufferMemory for full history retention
* Using ConversationBufferWindowMemory for sliding window context
* Inspecting and managing memory contents
* Clearing memory to reset conversations

### **Key Concepts Covered**

* Stateless vs stateful conversational AI
* Memory types in LangChain
* Token management and context window limits
* Conversation history storage and retrieval
* Trade-offs between memory completeness and efficiency

**Important:** For demonstration purposes and simplicity, we use a small model (0.5B parameters). These smaller models often do **not** follow instructions reliably. **For more consistent results, switch to a larger model or a private one such as GPT-5.**

## 1. Setup

Install dependencies and load the model:

In [ ]:
# Install required packages
%pip install langchain langchain-huggingface langchain-openai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 3.6 MB/s eta 0:00:00


In [ ]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Load Qwen 2.5 Instruct 0.5B model
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.7,
    return_full_text=False,
    do_sample=False # Do sample False (Is like 0 temperature)
)

llm = HuggingFacePipeline(pipeline=pipe)
print("✅ Model ready!")

# TO USE AN OPEN AI MODEL:
#from langchain_openai import ChatOpenAI
#import os

# Make sure your API key is set
#os.environ["OPENAI_API_KEY"] = "sk-"

#model_id = "gpt-4o-mini"

#llm = ChatOpenAI(
#    model=model_id,
#    temperature=0.1,
#)

Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Model ready!


## 2. Why Memory?

Without memory, each interaction is independent:

In [ ]:
from langchain.prompts import PromptTemplate

# Simple chain without memory
prompt = PromptTemplate(
    input_variables=["question"],
    template="Answer this question: {question}"
)

chain = prompt | llm

# First question
print("User: My name is David. How are you?")
response1 = chain.invoke({"question": "My name is David. How are you?"})
print(f"Bot: {response1}\n")

# Second question - it won't remember
print("User: What's my name?")
response2 = chain.invoke({"question": "What's my name?"})
print(f"Bot: {response2}")
print("\n❌ The model doesn't remember the previous interaction!")

User: My name is David. How are you?
Bot: content="Hello, David! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?" additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 20, 'total_tokens': 52, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_51db84afab', 'id': 'chatcmpl-CfTSymvGBvxo7SPDC2SnqmV3gz0wJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--645c1c87-9837-4780-b335-70715e8b5acb-0' usage_metadata={'input_tokens': 20, 'output_tokens': 32, 'total_tokens': 52, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}

User: What's my name?
Bot: content="I'm 

## 3. ConversationBufferMemory

Stores the entire conversation history:

In [ ]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

# Create memory
memory = ConversationBufferMemory()

# Create conversation chain with memory
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True  # Shows what's happening
)

# Have a conversation
print("=" * 50)
response1 = conversation.predict(input="Hi! My name is David and I love Python.")
print(f"\nBot: {response1}\n")

print("=" * 50)
response2 = conversation.predict(input="What's my name?")
print(f"\nBot: {response2}\n")

print("=" * 50)
response3 = conversation.predict(input="What programming language do I like?")
print(f"\nBot: {response3}")

/tmp/ipython-input-1773212044.py:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()
/tmp/ipython-input-1773212044.py:8: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :class:`~langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  conversation = ConversationChain(




> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi! My name is David and I love Python.
AI:

> Finished chain.

Bot: Hello, David! It's great to meet you! Python is such a versatile and powerful programming language. What do you love most about it? Is it the simplicity of the syntax, the vast libraries available, or perhaps the community support? There’s so much you can do with Python, from web development to data analysis and machine learning!



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it t

## 4. Viewing Memory Contents

You can inspect what's stored in memory:

In [ ]:
# View memory contents
print("Memory buffer:")
print(memory.buffer)
print("\n" + "=" * 50 + "\n")

# View as messages
print("Chat messages:")
print(memory.chat_memory.messages)

Memory buffer:
Human: Hi! My name is David and I love Python.
AI: Hello, David! It's great to meet you! Python is such a versatile and powerful programming language. What do you love most about it? Is it the simplicity of the syntax, the vast libraries available, or perhaps the community support? There’s so much you can do with Python, from web development to data analysis and machine learning!
Human: What's my name?
AI: Your name is David! It's nice to chat with you, David. Do you have any specific projects you're working on in Python right now?
Human: What programming language do I like?
AI: You like Python! It's a fantastic choice for many applications. Are there any particular areas of Python that you're especially interested in, like web development, data science, or automation?


Chat messages:
[HumanMessage(content='Hi! My name is David and I love Python.', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello, David! It's great to meet you! Python is such a vers

## 5. ConversationBufferWindowMemory

Only keeps the last K interactions (useful for long conversations):

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

# Keep only last 2 interactions (1 interaction = 1 user message + 1 AI response)
window_memory = ConversationBufferWindowMemory(k=2)

conversation_window = ConversationChain(
    llm=llm,
    memory=window_memory,
    verbose=False
)

# Have multiple interactions
print("Interaction 1:")
r1 = conversation_window.predict(input="My favorite color is blue")
print(f"Bot: {r1}\n")

print("Interaction 2:")
r2 = conversation_window.predict(input="I have a cat named Paco")
print(f"Bot: {r2}\n")

print("Interaction 3:")
r3 = conversation_window.predict(input="I work as a teacher")
print(f"Bot: {r3}\n")

# This should remember recent info but not the first
print("Interaction 4 - Testing memory:")
r4 = conversation_window.predict(input="What's my favorite color?")
print(f"Bot: {r4}")
print("\n⚠️ It may not remember the color (too old)!")

print("\nCurrent memory window:")
print(window_memory.buffer)

Interaction 1:


/tmp/ipython-input-520284851.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  window_memory = ConversationBufferWindowMemory(k=2)


Bot: That's great! Blue is such a calming and versatile color. It can represent tranquility, trust, and even creativity. There are so many shades of blue too, like navy, sky blue, and turquoise. Do you have a specific shade of blue that you like the most? Or perhaps a favorite item that’s blue?

Interaction 2:
Bot: Paco is such a fun name for a cat! Cats can have such unique personalities. What does Paco look like? Is he a specific breed or a mix? I’d love to hear more about his quirks or any funny stories you have about him!

Interaction 3:
Bot: That's wonderful! Teaching is such a rewarding profession. You get to shape young minds and inspire the next generation. What subject do you teach? And do you have a favorite lesson or activity that you enjoy doing with your students? I’d love to hear about your experiences in the classroom!

Interaction 4 - Testing memory:
Bot: I’m not sure what your favorite color is since you haven’t mentioned it yet! But I’d love to know! Do you have a col

## 8. Clearing Memory

You can reset the conversation:

In [ ]:
# Clear memory from earlier example
memory.clear()

print("Memory cleared!")
print(f"Buffer: '{memory.buffer}'")
print(f"Messages: {memory.chat_memory.messages}")

Memory cleared!
Buffer: ''
Messages: []
